In [0]:
from pyspark.sql import functions as F, Window

tr = spark.table("workspace.equipment.silver_train")
SENS = ["op1", "op2", "op3"] + [f"s{i}" for i in range(1, 22)]

d = tr.agg(*[F.countDistinct(c).alias(c) for c in SENS]).first().asDict()
keep = [c for c in SENS if d[c] > 1]
print("dropped constant columns:", [c for c in SENS if c not in keep])

w = Window.partitionBy("unit").orderBy("cycle").rowsBetween(-4, 0)

def feats(df):
    return df.select("*", *[F.avg(c).over(w).alias(f"{c}_avg5") for c in keep])

gt = feats(tr).withColumn("rul_capped", F.least(F.col("rul"), F.lit(125)))
gt.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.equipment.gold_train_features")

feats(spark.table("workspace.equipment.silver_test")).write.mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("workspace.equipment.gold_test_features")